### Célula 1: Parâmetros do Rastreador e Caminhos do Dataset

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath(".."))

# --- Caminhos do Dataset ---
DATASET_ROOT_FOLDER = '../../data/tiger2'
IMAGE_FOLDER = os.path.join(DATASET_ROOT_FOLDER, 'imgs')
GT_TXT_PATH = os.path.join(DATASET_ROOT_FOLDER, 'tiger2_gt.txt')

In [ ]:
from tracker.cluswisard_tracker import ClusWisardTracker

parameters = {
    'INPUT_PATTERN_SIDE': 32,
    'SEARCH_WINDOW_SCALE': 1.5,
    'NUM_SEARCH_SAMPLES': 500,
    'CLUSAZARD_ADDRESS_SIZE': 2,
    'CLUSAZARD_MIN_SCORE': 0.8,
    'CLUSAZARD_THRESHOLD': 5,
    'CLUSAZARD_DISCRIMINATOR_LIMIT': 5,
    'CLUSAZARD_BLEACHING_ACTIVATED': False,
    'CLUSAZARD_ACTIVATION_DEGREE': True,
    'CLUSAZARD_RETURN_CONFIDENCE': False,
    'CLUSAZARD_CLASSES_DEGREES': False
}

tracker = ClusWisardTracker(IMAGE_FOLDER, GT_TXT_PATH, parameters)
resultados = tracker.track()


In [ ]:
print(resultados)

In [ ]:
import numpy as np
from collections import deque

def architecture_1(frames, init_x, init_y, patch_size=8, search_radius=4, threshold=0.7, max_discriminators=3):
    # fila de discriminadores (deque para facilitar remoção do final)
    discriminators = deque()
    
    # 1 e 2: Treinar primeiro discriminador no patch inicial
    first_patch = extract_patch(frames[0], init_x, init_y, patch_size)
    d = ClusWisard()
    d.train(first_patch)
    discriminators.appendleft(d)  # 3: adiciona na fila
    
    object_pos = (init_x, init_y)
    
    # Para cada frame seguinte
    for frame_idx in range(1, len(frames)):
        frame = frames[frame_idx]
        best_score = -1
        best_pos = None
        
        # 5: busca no entorno do último objeto conhecido
        for dx in range(-search_radius, search_radius+1):
            for dy in range(-search_radius, search_radius+1):
                candidate_x = object_pos[0] + dx
                candidate_y = object_pos[1] + dy
                patch = extract_patch(frame, candidate_x, candidate_y, patch_size)
                
                # avaliar com todos discriminadores, pega maior score
                scores = [d.classify(patch) for d in discriminators]
                max_score = max(scores) if scores else 0
                
                if max_score > best_score:
                    best_score = max_score
                    best_pos = (candidate_x, candidate_y)
        
        print(f"Frame {frame_idx}: Melhor score = {best_score:.2f} na pos {best_pos}")
        
        # 7: Se a maior pontuação for menor que o limiar, treina novo discriminador
        if best_score < threshold:
            new_patch = extract_patch(frame, best_pos[0], best_pos[1], patch_size)
            new_discriminator = ClusWisard()
            new_discriminator.train(new_patch)
            
            # 9-11: fila cheia? remove último
            if len(discriminators) >= max_discriminators:
                removed = discriminators.pop()
                print("Fila cheia. Removendo discriminador mais antigo.")
            # 12: adiciona novo no início
            discriminators.appendleft(new_discriminator)
            print("Novo discriminador treinado e adicionado na fila.")
        
        object_pos = best_pos
    
    return discriminators


# Executa arquitetura 1
discriminators = architecture_1(frames, init_x=15, init_y=15)

print(f"Total discriminadores treinados: {len(discriminators)}")
